In [ ]:
import pandas as pd
from pathlib import Path

OUT_ROOT = Path("./rosetta_batch_out")

def read_one_interface_score(score_file):
    df = pd.read_csv(score_file, sep=r"\s+", comment="#")
    return df

def gather_all_scores(out_root):
    all_rows = []

    for complex_dir in sorted(out_root.iterdir()):
        if not complex_dir.is_dir():
            continue

        complex_id = complex_dir.name

        for sample_dir in sorted(complex_dir.iterdir()):
            if not sample_dir.is_dir():
                continue

            sample_name = sample_dir.name
            interface_dir = sample_dir / "interface"

            if not interface_dir.exists():
                continue

            for score_file in sorted(interface_dir.glob("*.interface.sc")):
                try:
                    df = read_one_interface_score(score_file)

                    if df.empty:
                        continue

                    df["complex"] = complex_id
                    df["sample"] = sample_name
                    df["score_file"] = score_file.name


                    df["decoy"] = score_file.stem.replace(".interface", "")

                    all_rows.append(df)

                except Exception as e:
                    print(f"Failed to read {score_file}: {e}")

    if all_rows:
        result = pd.concat(all_rows, ignore_index=True)
        return result
    else:
        return pd.DataFrame()

all_scores_df = gather_all_scores(OUT_ROOT)

print(all_scores_df.shape)
all_scores_df.head()


(21285, 5)


,SEQUENCE:,complex,sample,score_file,decoy
0,description,10558,sample_0,sample_0_0001.interface.sc,sample_0_0001
1,sample_0_0001_0001,10558,sample_0,sample_0_0001.interface.sc,sample_0_0001
2,sample_0_0001_0001,10558,sample_0,sample_0_0001.interface.sc,sample_0_0001
3,description,10558,sample_0,sample_0_0002.interface.sc,sample_0_0002
4,sample_0_0002_0001,10558,sample_0,sample_0_0002.interface.sc,sample_0_0002


In [3]:
print(all_scores_df.columns)

Index(['SEQUENCE:', 'complex', 'sample', 'score_file', 'decoy'], dtype='object')


In [ ]:
import pandas as pd
from pathlib import Path

OUT_ROOT = Path("./rosetta_batch_out")

def read_one_interface_score(score_file):

    df = pd.read_csv(score_file, sep=r"\s+", skiprows=1)


    if "SCORE:" in df.columns:
        df = df.drop(columns=["SCORE:"])


    if "description" in df.columns:
        df = df[df["description"] != "description"].copy()

    return df

def gather_all_scores(out_root):
    all_rows = []

    for complex_dir in sorted(out_root.iterdir()):
        if not complex_dir.is_dir():
            continue

        complex_id = complex_dir.name

        for sample_dir in sorted(complex_dir.iterdir()):
            if not sample_dir.is_dir():
                continue

            sample_name = sample_dir.name
            interface_dir = sample_dir / "interface"

            if not interface_dir.exists():
                continue

            for score_file in sorted(interface_dir.glob("*.interface.sc")):
                try:
                    df = read_one_interface_score(score_file)

                    if df.empty:
                        continue

                    df["complex"] = complex_id
                    df["sample"] = sample_name
                    df["score_file"] = score_file.name
                    df["decoy"] = score_file.stem.replace(".interface", "")

                    all_rows.append(df)

                except Exception as e:
                    print(f"Failed to read {score_file}: {e}")

    if all_rows:
        result = pd.concat(all_rows, ignore_index=True)
        return result
    else:
        return pd.DataFrame()

all_scores_df = gather_all_scores(OUT_ROOT)

print(all_scores_df.shape)
print(all_scores_df.columns.tolist())
print(all_scores_df[["complex", "sample", "decoy", "total_score", "dG_cross", "dG_separated"]].head(20))


(11385, 45)
['total_score', 'complex_normalized', 'dG_cross', 'dG_cross/dSASAx100', 'dG_separated', 'dG_separated/dSASAx100', 'dSASA_hphobic', 'dSASA_int', 'dSASA_polar', 'delta_unsatHbonds', 'dslf_fa13', 'fa_atr', 'fa_dun', 'fa_elec', 'fa_intra_rep', 'fa_intra_sol_xover4', 'fa_rep', 'fa_sol', 'hbond_E_fraction', 'hbond_bb_sc', 'hbond_lr_bb', 'hbond_sc', 'hbond_sr_bb', 'hbonds_int', 'lk_ball_wtd', 'nres_all', 'nres_int', 'omega', 'p_aa_pp', 'packstat', 'per_residue_energy_int', 'pro_close', 'rama_prepro', 'ref', 'sc_value', 'side1_normalized', 'side1_score', 'side2_normalized', 'side2_score', 'yhh_planarity', 'description', 'complex', 'sample', 'score_file', 'decoy']
   complex    sample          decoy  total_score  dG_cross  dG_separated
0    10558  sample_0  sample_0_0001       26.412     3.633         5.160
1    10558  sample_0  sample_0_0001       26.453     3.655         5.182
2    10558  sample_0  sample_0_0002       26.464     3.655         5.189
3    10558  sample_0  sample_0_0

In [5]:
all_scores_df.to_csv("a.csv")